# Урок 5.7. Закрепление: квиз и FastAPI-прокси перед Ollama

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 7 из 7 · финал

Шесть уроков назад Ollama была чёрным ящиком «поставил и работает». Сегодня —
финальная сборка: всё, чему научил модуль, собрано в одно приложение
(`ollama-proxy/`), плюс сводный квиз по сценариям из реальной практики.

## 🎯 Что в финале

- Карта модуля: шесть уроков в одной таблице с главными числами
- Разбор `ollama-proxy/` — какие решения из каких уроков в нём собраны
- Живой запуск тестов (42 теста на фейках — Ollama не нужна) и самого прокси
- Сводный квиз: 10 сценарных вопросов
- Практика на три уровня сложности — эталонные решения уже в проекте


## Карта модуля

| Урок | Вопрос | Что запомнить (числа — с моей машины) |
|---|---|---|
| 5.1 | Когда локально? | Жёсткие ограничения → матрица. Потолок CPU-сервера ~10 млн ток/мес; с бюджетными облачными ценами экономика включается на десятках тысяч запросов/день, с топовыми — на сотнях |
| 5.2 | Какую модель? | Q4_K_M — дефолт; скорость ≈ полоса RAM / размер × КПД (~40% на ноутбуке); свой бенчмарк: instruct 10/10 против base 2/10 (`done_reason=length`); кеш префикса надувает prefill-замеры |
| 5.3 | Как сделать сервисом? | NSSM/systemd/Docker; keep_alive против холодного старта (3.3 с → 0.01 с); матрица retry; `127.0.0.1`, не localhost; конфиг демона — в server.log |
| 5.4 | А без демона? | llama-cpp-python: блоб Ollama = готовый GGUF; `n_ctx` по умолчанию 512!; грамматики GBNF гарантируют формат, но требуют согласованного промпта |
| 5.5 | Как наблюдать? | Пирамида health-check (L3 деструктивен!); метрики с каждого ответа; baseline 83 ток/с → 18 при «съезде» с GPU = алерт; for-duration против ложных тревог |
| 5.6 | Как чинить? | KV-кеш: +1.2 ГБ от num_ctx 32k; лесенка частичной выгрузки 68→34→25→18 ток/с; num_predict всегда; очередь сериализует (2.7/3.7/4.7 с); digest вместо тега |

## Зачем прокси, если Ollama и так работает

Вспомните обещание из урока 5.3: «наружу Ollama выставляют только через
reverse-proxy с auth — или наш прокси из урока 5.7». Пришло время его исполнить.
У демона нет аутентификации, нет потолков на запрос, нет метрик по клиентам —
и чинить это в каждом клиенте по отдельности нельзя. Прокси — единственный вход:

```
клиенты --(X-API-Token)--> ollama-proxy :8005 --> Ollama :11434 --> модель
                            │ 401 без токена           (урок 5.3: auth нет)
                            │ 429 + Retry-After        (практика 5.7: лимит на токен)
                            │ потолок num_predict      (урок 5.6)
                            │ семафор + queue_ms       (уроки 3.2, 5.6)
                            │ retry по матрице         (урок 5.3)
                            │ окно ток/с + /metrics    (урок 5.5)
                            │ основная/запасная модель (капстоун 5.7)
                            └ healthz + доля GPU + digest (уроки 5.5, 5.6)
```

Это тот же паттерн, что API-gateway перед микросервисами: сквозные заботы
(auth, лимиты, наблюдаемость) живут в одном месте, а не размазаны по клиентам.

## Архитектура: 6 модулей + тесты

```
ollama-proxy/
├── app/
│   ├── config.py         # env-настройки — стиль урока 1.4
│   ├── ollama_client.py  # async-клиент: retry-матрица урока 5.3
│   ├── metrics.py        # TpsWindow из урока 5.5 + Prometheus, окна по моделям
│   ├── ratelimit.py      # лимит на токен: скользящее окно, Retry-After (практика)
│   ├── routing.py        # основная/запасная модель, канарейки (капстоун)
│   └── main.py           # create_app(client=..., clock=...) — DI как в mini-project 3.8
└── tests/                # фейки со счётчиками: проверяем и НЕ-вызовы
```

Главное конструктивное решение — то же, что в модуле 3: **зависимость приходит
снаружи**. `create_app(client=FakeOllama())` в тестах, `create_app()` в бою.
Фейк считает вызовы, поэтому тесты умеют проверять отрицания: «после ReadTimeout
прокси НЕ пошёл в Ollama второй раз» — это и есть матрица retry, закреплённая тестом.

Вторая находка — `httpx.MockTransport`: retry-логику настоящего `OllamaClient`
мы тестируем без сети, подсовывая транспорт, который сам отвечает 503 или бросает
`ConnectError`. Быстро, детерминированно, без Ollama.

Третья — **часы инжектируются** (`create_app(clock=...)`): лимит «N запросов
в минуту» и сброс его окна тестируются переводом фейковых часов, а не `sleep(60)`.
Это тот же приём, что `Alert` с инъекцией `now` в уроке 5.5.


## Запускаем тесты

42 теста, ноль обращений к сети и ни одного `sleep` — прогоним прямо отсюда (нужны зависимости
из `ollama-proxy/requirements.txt`):


In [ ]:
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--no-header", "-o", "addopts="],
    cwd="ollama-proxy", capture_output=True, text=True,
    timeout=300, encoding="utf-8", errors="replace",
)
lines = proc.stdout.splitlines()
for line in lines:
    if "::" in line or "passed" in line or "failed" in line:
        print(line)

Обратите внимание на группы:

- `test_client.py` — матрица retry через MockTransport: 503×2 → успех с третьей
  попытки; 404 и ReadTimeout — ровно один вызов, без повторов;
- `test_api.py` — 401 без токена (и счётчик подтверждает: до Ollama запрос
  не дошёл), потолок `num_predict` срезает и 100000, и «не задано»;
  504 после ReadTimeout; healthz `down`/`degraded` (частичный GPU, дрейф digest);
- `test_metrics.py` — перцентили окна и холодные старты;
- `test_models_api.py` — `/models`: размеры, digest, роли primary/fallback, 401 без
  токена, 502 при мёртвой Ollama, пустой диск;
- `test_ratelimit.py` — окно лимита: превышение с честным `Retry-After` (60 → 50 с
  после перевода часов на 10 с), сброс окна, независимость токенов; и HTTP-контракт:
  429 не доходит до Ollama, чужой токен получает 401 и ничей лимит не тратит;
- `test_routing.py` — деградация туда и обратно: по доле GPU (40% → запасная,
  100% → основная) и по p50 (20 ток/с при baseline 80 → запасная; три канарейки
  по 80 ток/с → основная). Роутер и HTTP-цикл проверены отдельно.

## Живой запуск: прокси честен даже без Ollama

Поднимем прокси как отдельный процесс. Маленькое признание: в момент сборки
этого урока Ollama на моей машине **выключена, а модели удалены** — модуль
закончен, я прибрался. Идеальный момент проверить, как прокси ведёт себя
в аварии: сервис перед мёртвым апстримом обязан отвечать быстро и честно,
а не зависать. Ячейка ниже работает в обоих мирах — с Ollama и без:


In [ ]:
import time

import httpx

PROXY = "http://127.0.0.1:8005"

proxy_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8005"],
    cwd="ollama-proxy", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
proxy_up = False
# Таймаут пробы 5 с: healthz внутри сам ходит к Ollama, и при выключенном демоне
# отказ соединения на Windows может занимать ~2 с (мы намерили это в уроке 5.3)
with httpx.Client(timeout=5) as probe:   # один клиент на все пробы (урок 5.3)
    for _ in range(120):                 # ждём старт uvicorn до ~2 минут
        try:
            probe.get(f"{PROXY}/healthz")
            proxy_up = True
            break
        except httpx.HTTPError:
            time.sleep(0.5)
if not proxy_up:
    proxy_proc.terminate()
    raise RuntimeError("прокси не поднялся: запустите uvicorn руками и посмотрите вывод")

health = httpx.get(f"{PROXY}/healthz", timeout=10).json()
print("healthz:", health)

print("\nбез токена:")
r = httpx.post(f"{PROXY}/generate", json={"prompt": "hi"}, timeout=10)
print(f"  HTTP {r.status_code}: {r.json()['detail']}")

if health["status"] == "down":
    print("\nOllama выключена — смотрим аварийное поведение с токеном:")
    t0 = time.perf_counter()
    r = httpx.post(f"{PROXY}/generate", json={"prompt": "hi"},
                   headers={"X-API-Token": "dev-token"}, timeout=60)
    print(f"  HTTP {r.status_code} за {time.perf_counter() - t0:.1f} с: {r.json()['detail']}")
    print("  (это 3 попытки подключения с backoff — матрица урока 5.3, не зависание)")
    print("\nметрики честно считают аварию:")
    for line in httpx.get(f"{PROXY}/metrics", timeout=10).text.splitlines():
        if line.startswith(("proxy_requests_total", "proxy_ollama_up")):
            print(" ", line)
else:
    print("\nOllama жива — аварийную ветку пропускаем, переходите к следующей ячейке.")

Мой прогон (Ollama выключена): healthz мгновенно ответил
`{"status": "down", "reason": "Ollama не отвечает…"}`, запрос без токена — `401`
(до апстрима даже не дошли), а с токеном — `502` через ~7–8 секунд: три попытки
подключения с backoff и джиттером, после чего честная ошибка. Ни одного зависшего
клиента, и в `/metrics` появился `proxy_requests_total{status="upstream_down"}`.

Сравните с поведением «клиенты ходят в Ollama напрямую»: каждый ждал бы свой
таймаут, никто бы не посчитал аварию, и никакой 401 не защищал бы порт.

## Счастливый путь (нужна Ollama + qwen2.5:3b)

Если у вас демон запущен и модель на месте (`ollama pull qwen2.5:3b`, уроки 1.5
и 5.3), следующая ячейка покажет полный цикл. Поля ответа вы уже знаете:
`tokens_per_second` — скорость из телеметрии (урок 5.5), `load_seconds` —
холодный старт (урок 5.3), `queue_ms` — время в очереди (урок 5.6).


In [ ]:
health = httpx.get(f"{PROXY}/healthz", timeout=10).json()
if health["status"] != "down":
    for label in ("первый запрос (возможен холодный старт)", "второй запрос (тёплый)"):
        t0 = time.perf_counter()
        r = httpx.post(f"{PROXY}/generate",
                       json={"prompt": "Что такое очередь сообщений? Одно предложение.",
                             "num_predict": 60, "temperature": 0.2},
                       headers={"X-API-Token": "dev-token"}, timeout=300)
        body = r.json()
        print(f"{label}: HTTP {r.status_code} за {time.perf_counter() - t0:.1f} с")
        if r.status_code == 200:
            print(f"  текст: {body['text'][:80]}…")
            print(f"  {body['completion_tokens']} ток @ {body['tokens_per_second']:.1f} ток/с, "
                  f"загрузка {body['load_seconds']:.2f} с, очередь {body['queue_ms']} мс")
    print("\nhealthz после запросов:", httpx.get(f"{PROXY}/healthz", timeout=10).json())
else:
    print("Пропущено: Ollama не запущена. Запустите её, скачайте qwen2.5:3b "
          "и перезапустите ячейки урока — прокси покажет полный цикл.")

In [ ]:
# Прибираем за собой: останавливаем прокси
proxy_proc.terminate()
proxy_proc.wait(timeout=10)
print("прокси остановлен")

## Деплой

Прокси разворачивается теми же способами, что Ollama в уроке 5.3:

- **venv + uvicorn** под NSSM/systemd — команды в `ollama-proxy/README.md`;
- **Docker**: `docker compose up -d --build` в `ollama-proxy/` поднимает пару
  Ollama+прокси, причём порт Ollama наружу **не публикуется** — вход только через
  прокси с токеном (compose в уроке не прогонялся — проверьте сами);
- не забудьте watchdog из урока 5.5 рядом: прокси мониторит Ollama,
  а кто мониторит прокси?

## Сводный квиз модуля

<details><summary><b>1. Стартап (200 диалогов/день) хочет «развернуть локальную LLM, чтобы сэкономить». Что ответите?</b></summary>

Посчитать: такой трафик на бюджетной облачной модели стоит ~$2/мес против ~$102/мес
за свой CPU-сервер (урок 5.1). Экономика включается на десятках тысяч запросов
в день (или на топовых облачных ценах). Локально здесь имеет смысл только ради
приватности, комплаенса или offline — и это надо назвать прямо, а не прятать
за «сэкономим».
</details>

<details><summary><b>2. Коллега выбрал модель по первому месту в лидерборде. Какие три возражения?</b></summary>

Контаминация (тесты утекают в обучение), англоцентричность (русский может хромать),
и главное — ваша задача в лидерборде не представлена. Решение из урока 5.2:
свой мини-бенчмарк из 10–30 задач с автопроверкой; наш живой прогон дал instruct
10/10 против base 2/10 — такие вещи лидерборд не покажет.
</details>

<details><summary><b>3. На 8 ГБ RAM: 3B в Q8_0 или 8B в Q4_K_M? И почему не F16 «для качества»?</b></summary>

По качеству «большая в Q4» обычно бьёт «маленькую в Q8», но 8B-Q4 съест ~6–7 ГБ —
впритык к swap. Для отзывчивой машины разумнее 3B, и именно Q4: у Q8 качество
почти то же, а читать из RAM вдвое больше — вдвое медленнее (урок 5.2). F16 на CPU —
вчетверо медленнее Q4 при неизмеримом приросте.
</details>

<details><summary><b>4. Ваш бенчмарк показал prefill 5000 ток/с на CPU. Радоваться?</b></summary>

Проверить промпты: скорее всего, они повторяются, и вы измерили **кеш префикса**,
а не модель (prefill-детектив урока 5.2). Честный CPU-prefill на уникальных
промптах — ~50–60 ток/с, лишь в 3–4 раза быстрее генерации. Скорость меряют
уникальными промптами.
</details>

<details><summary><b>5. После ночного рестарта демона первый утренний запрос идёт 30 секунд, дальше всё быстро. Диагноз и лечение?</b></summary>

Холодный старт: модель грузилась с диска (поле `load_duration` это покажет).
Лечение из урока 5.3: подобрать `keep_alive` под трафик и добавить прогрев
после рестарта (запрос с `num_predict: 1` из скрипта деплоя) — тогда холодный
старт платит деплой, а не первый пользователь.
</details>

<details><summary><b>6. Клиенты стали получать 503 от Ollama. Это авария?</b></summary>

Это **защита**: очередь `OLLAMA_MAX_QUEUE` полна, демон честно отказывает вместо
бесконечного ожидания (урок 5.3). Клиенты должны повторить с backoff и джиттером.
Алерт нужен, если 503 систематические — тогда это разговор о ёмкости: вторая нода,
меньшая модель или пересмотр потолка запросов (урок 5.6).
</details>

<details><summary><b>7. Нужен ответ строго одним из пяти статусов. Промпт с retry или грамматика? Какая ловушка?</b></summary>

Грамматика GBNF (урок 5.4): сэмплер физически не выдаст токен вне списка —
это сильнее любых уговоров и retry. Ловушка: грамматика умеет запрещать,
а не подсказывать — без «ответь ровно одним словом» модель, зажатая маской,
может выбирать одинаковый бессмысленный вариант для всех входов (мы получали
`low` на всё подряд).
</details>

<details><summary><b>8. Какую проверку поставить балансировщику: /api/version или генерацию одного токена?</b></summary>

`/api/version` (или наш `/healthz`): дешёвая, можно дёргать часто. Генерация —
деструктивная проверка L3 (урок 5.5): грузит модель, сбрасывает keep_alive,
занимает слот — частые пробы сами создают нагрузку. L3 — редкой пробой watchdog'а,
и минимум на 8 токенов (на одном токене скорость — бессмыслица, мы получали
«миллион ток/с»).
</details>

<details><summary><b>9. В пятницу сделали ollama pull, в понедельник «модель поглупела». Как надо было?</b></summary>

Тег — указатель, не версия (урок 5.6). Правильная процедура: заморозить прод-тег
(`ollama cp`), записать digest в конфиг, после pull сверить digest и прогнать
канареечный бенчмарк урока 5.2; перевешивать прод-тег только при непросевшем
счёте. И помнить, что демон тоже самообновляется (мы это пережили в уроке 3.2).
</details>

<details><summary><b>10. Зачем прокси, если клиенты могут ходить в Ollama напрямую?</b></summary>

У Ollama нет аутентификации, потолков на запрос и метрик по клиентам. Прокси —
единственный вход, где один раз решаются сквозные заботы: 401 без токена, потолок
`num_predict`, очередь с честным `queue_ms`, retry по матрице, окно скоростей
и `/metrics`, healthz с долей GPU и дрейфом digest. Наш живой тест: даже
с выключенной Ollama прокси отвечает быстро и честно, а не зависает.
</details>

## Практика: три уровня

Все три уровня реализованы в проекте как эталон — `GET /models` в `main.py`,
`app/ratelimit.py`, `app/routing.py` и тесты `test_models_api.py`,
`test_ratelimit.py`, `test_routing.py`. Но сначала попробуйте сами по формулировкам
ниже, а потом сверьтесь. В формулировках не видны три решения, которые пришлось
принять по дороге: **скользящее окно вместо счётчика-по-минутам** (счётчик
пропускает 2N запросов на стыке минут и не умеет честный `Retry-After`),
**check-and-consume одним шагом** (между «проверить» и «учесть» пролезает второй
запрос того же токена) и **канарейки при деградации** (пока трафик идёт в запасную,
основная молчит, её p50 не обновляется — и вернуться было бы не по чему).

1. **Базовый — `/models`.** Добавьте эндпоинт, который отдаёт список моделей
   с диска Ollama (`/api/tags`) с их размерами и digest. Плюс тест на фейке.
2. **Средний — лимиты на токен.** Каждому токену — не больше N запросов
   в минуту: счётчик в памяти, при превышении — 429 с заголовком `Retry-After`.
   Тесты: превышение, сброс окна, независимость токенов.
3. **Капстоун — две модели и деградация.** Прокси знает `qwen2.5:3b` (основная)
   и лёгкую запасную (например, `qwen2.5:1.5b`). Если healthz видит, что основная
   съехала с GPU или p50 упал ниже половины baseline, — новые запросы идут
   в запасную, ответ помечается полем `"degraded_to"`. Тесты на фейках
   покрывают переключение туда и обратно.

## Итоги модуля 5

- **Решение «локально или API»** принимается матрицей после жёстких ограничений,
  а деньги считаются в обе стороны — с потолком производительности CPU (5.1).
- **Модель выбирает ваш бенчмарк**, а не лидерборд; квантизация Q4_K_M — дефолт;
  скорость диктует полоса памяти, и формулы честны только с реальным КПД
  и уникальными промптами (5.2).
- **Ollama — это демон**, и обращаться с ним надо как с сервисом: супервизор,
  keep_alive, очередь, матрица retry, конфиг из server.log (5.3).
- **Прямой inference** даёт контроль (грамматики!), Ollama — обслуживание многих;
  блоб Ollama = готовый GGUF (5.4).
- **Наблюдаемость**: метрики с каждого ответа, baseline машины, деструктивность
  глубоких проб, for-duration (5.5). **Диагностика**: KV-кеш, частичная выгрузка,
  num_predict, digest (5.6).
- И всё это собирается в ~750 строк прокси с 42 тестами (вместе с тремя
  уровнями практики) — потому что каждое решение уже было отработано в своём уроке.

**Дальше**: модуль 6 — дообучение (fine-tuning): когда промптов и RAG мало,
учим модель на своих данных через LoRA — бесплатно, на Colab.

## Полезные ссылки

- FastAPI: <https://fastapi.tiangolo.com>
- httpx (MockTransport): <https://www.python-httpx.org/advanced/transports/>
- prometheus-client: <https://prometheus.github.io/client_python/>
- Ollama API: <https://github.com/ollama/ollama/blob/main/docs/api.md>
